In [1]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,image_path,label
0,output_dataset/train/000000.png,violin
1,output_dataset/train/000001.png,moon
2,output_dataset/train/000002.png,violin
3,output_dataset/train/000003.png,book
4,output_dataset/train/000004.png,tree


In [2]:
from sklearn.preprocessing import LabelEncoder
enc = LabelEncoder()
df_train.label = enc.fit_transform(df_train.label)

df_train.head()

,image_path,label
0,output_dataset/train/000000.png,19
1,output_dataset/train/000001.png,13
2,output_dataset/train/000002.png,19
3,output_dataset/train/000003.png,3
4,output_dataset/train/000004.png,17


In [3]:
df_train.label.nunique()

20

In [50]:
import torch

epochs = 10
learning_rate = 0.0001
batch_size = 32

device = torch.device("cuda")
device

device(type='cuda')

In [51]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image

train_transforms = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

valid_transforms = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class MyDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        path = self.df.image_path.iloc[idx]
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        label = torch.tensor(self.df.label.iloc[idx], dtype=torch.int64)
        return img, label

from sklearn.model_selection import train_test_split
df_1, df_2 = train_test_split(df_train, test_size=0.2, random_state=42)

train_dataset = MyDataset(df_1, train_transforms)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

valid_dataset = MyDataset(df_2, valid_transforms)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size)

train_dataset[0][0][0][0][0]

tensor(1.6667)

In [52]:
import torch.nn as nn
import torchvision.models

model = torchvision.models.resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Sequential(
    nn.Linear(512, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 20)
)

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [53]:
def train(model, train_loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        loss = criterion(model(inputs), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss
    print(f"Epoch {epoch}")
    print(f"Training loss {total_loss/len(train_loader)}")

def test(model, test_loader, criterion, epoch):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            loss = criterion(model(inputs), labels)
            total_loss += loss
    print(f"Valdiation loss {total_loss/len(test_loader)}")

In [54]:
for epoch in range(1, epochs+1):
    train(model, train_loader, criterion, optimizer, epoch)
    test(model, valid_loader, criterion, epoch)

Epoch 1
Training loss 2.8297226428985596
Valdiation loss 2.726099729537964
Epoch 2
Training loss 2.1659603118896484
Valdiation loss 2.2484021186828613
Epoch 3
Training loss 1.6473032236099243
Valdiation loss 1.9144022464752197
Epoch 4
Training loss 1.2407954931259155
Valdiation loss 1.634188175201416
Epoch 5
Training loss 0.9162792563438416
Valdiation loss 1.3893483877182007
Epoch 6
Training loss 0.6844932436943054
Valdiation loss 1.1684720516204834
Epoch 7
Training loss 0.4760718047618866
Valdiation loss 0.9621003866195679
Epoch 8
Training loss 0.31335753202438354
Valdiation loss 0.8086087703704834
Epoch 9
Training loss 0.2066437304019928
Valdiation loss 0.7361385226249695
Epoch 10
Training loss 0.15714703500270844
Valdiation loss 0.7057991027832031


In [55]:
final_transforms = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class FinalDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        path = self.df.image_path.iloc[idx]
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        return img

final_dataset = FinalDataset(df_test, final_transforms)
final_loader = DataLoader(final_dataset, batch_size=batch_size)

final_dataset[0][0][0][0]

tensor(1.4269)

In [56]:
model.eval()
preds = []

with torch.no_grad():
    for inputs in final_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        outputs = torch.argmax(outputs, dim=1)
        preds.extend(outputs.cpu().numpy())

preds[:10]

[17, 6, 16, 7, 17, 6, 5, 5, 11, 19]

In [57]:
preds = enc.inverse_transform(preds)

In [58]:
df_preds = pd.DataFrame({
    "image_path": df_test.image_path,
    "label": preds
})

df_preds

,image_path,label
0,output_dataset/test/000000.png,tree
1,output_dataset/test/000001.png,dog
2,output_dataset/test/000002.png,sun
3,output_dataset/test/000003.png,elephant
4,output_dataset/test/000004.png,tree
...,...,...
95,output_dataset/test/000095.png,boat
96,output_dataset/test/000096.png,elephant
97,output_dataset/test/000097.png,flower
98,output_dataset/test/000098.png,tree


In [59]:
df_preds.to_csv("submission.csv", index=False)